# E7 — Leave-One-Source-Out: đo domain gap GIỮA các nguồn của PPE-Unified (YOLO26s, 15 epochs/run)

**Câu hỏi:** E3 đã đo domain gap ra NGOÀI benchmark (SH17: −70 đến −88 điểm). Vậy domain gap BÊN TRONG benchmark
— giữa 3 nguồn A (helmet), B (vest/boots), C (gloves/goggles/mask) — lớn đến đâu? Trả lời câu này định lượng được
mức phụ thuộc "phong cách ảnh nguồn" của từng lớp, bổ sung trực tiếp cho phân tích domain bias mục 7.2/7.4 của báo cáo.

**Thiết kế (3 run, quay vòng):**

| Run | Train + Val | Test (zero-shot) | Lớp đo được |
|---|---|---|---|
| `loso_A` | B + C | toàn bộ ảnh nguồn A | helmet, no_helmet |
| `loso_B` | A + C | toàn bộ ảnh nguồn B | vest/no_vest, boots/no_boots (độc quyền B) + helmet |
| `loso_C` | A + B | toàn bộ ảnh nguồn C | gloves/goggles/mask các cặp (chủ lực C) |

- Lọc ảnh theo cột `source` của `manifest.csv`; train/val dùng đúng split gốc của 2 nguồn giữ lại (không re-split);
  test = TOÀN BỘ ảnh nguồn bị loại (mọi split — hợp lệ vì model chưa từng thấy nguồn đó, không có leakage).
- Chỉ chấm AP những lớp có ≥50 box trong tập test của run (lớp vắng mặt → NaN, không tính vào trung bình).
- Cùng YOLO26s seed 42, lr0 0,01, batch 16, 15 epochs, RFS tắt.

**Đầu ra:** `results/e7_loso.csv` — mỗi run: mAP@50 (các lớp đủ box), AP từng lớp chính, chênh so với in-distribution.
Kỳ vọng diễn giải: lớp chỉ có ở 1 nguồn (vest, boots) sẽ sụt sâu nhất — bằng chứng "độ đa dạng nguồn > số lượng mẫu" (Q3).

**Add Input:** `ppe-unified-v1`. GPU T4 x2 · Internet ON. Ước tính ~5-6h (3 run). Có checkpoint-resume.

In [ ]:
KEY_PREFIX = "e7_"
# ==================== CONFIG ====================
QUICK_TEST = False             # True = chạy thử 1 epoch/vài batch để kiểm tra pipeline; số liệu quick KHÔNG đưa vào báo cáo
SEED = 42

import os, random, shutil, glob, json, time
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)
STATE_PATH = f"{WORK}/state.json"
# RESUME: tự nạp state.json từ output phiên trước của CHÍNH notebook này (nếu Add Input)
STATE = {}
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any(k.startswith(KEY_PREFIX) for k in _s): STATE.update(_s); print("Resumed state:", list(_s))
def save_state():
    json.dump(STATE, open(STATE_PATH, "w"), ensure_ascii=False, indent=1)
def done(key): return key in STATE
random.seed(SEED)


In [ ]:
%pip install -q ultralytics
import torch; assert torch.cuda.is_available(), "Bật GPU T4 x2!"
print("GPU:", torch.cuda.get_device_name(0), "- OK")


In [ ]:
from pathlib import Path
import yaml, pandas as pd

def find_dataset_root():
    """Tìm PPE_Unified_v1 trong /kaggle/input (glob đệ quy — không khóa path)."""
    for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
        root = os.path.dirname(c)
        if all(os.path.isdir(f"{root}/{s}/images") for s in ("train", "val", "test")):
            return root
    raise AssertionError("Chưa Add Input dataset ppe-unified-v1!")

def find_manifest():
    """manifest.csv truy vết nguồn gốc (source A/B/C) của từng ảnh — sinh bởi build_unified_dataset.py."""
    for c in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
        return pd.read_csv(c)
    raise AssertionError("Không thấy manifest.csv trong dataset input!")

DS_ROOT = find_dataset_root()
MANIFEST = find_manifest()
print("Dataset:", DS_ROOT, "| manifest:", len(MANIFEST), "dòng")
print(MANIFEST["source"].value_counts() if "source" in MANIFEST.columns else MANIFEST.columns.tolist())


In [ ]:
# ---------- Dựng dataset LOSO ----------
def build_loso(dst_name, held_out):
    """Train/val = 2 nguồn còn lại (giữ split gốc); test = TOÀN BỘ ảnh nguồn held_out."""
    dst = f"{WORK}/{dst_name}"
    if os.path.isdir(dst): shutil.rmtree(dst)
    for sp in ("train", "val", "test"):
        for sub in ("images", "labels"):
            os.makedirs(f"{dst}/{sp}/{sub}", exist_ok=True)
    for _, row in MANIFEST.iterrows():
        src_letter = str(row["source"]).upper()[0]
        split, fname = row["split"], row["file"]
        lbl = fname.rsplit(".", 1)[0] + ".txt"
        # nguồn bị loại → dồn hết vào test; nguồn giữ lại → theo split gốc nhưng bỏ test-split
        # (test-split của nguồn giữ lại không dùng — giữ nguyên tắc test chỉ chạm 1 lần ở notebook chính)
        if src_letter == held_out:
            tgt = "test"
        elif split in ("train", "val"):
            tgt = split
        else:
            continue
        shutil.copy(f"{DS_ROOT}/{split}/images/{fname}", f"{dst}/{tgt}/images/{fname}")
        shutil.copy(f"{DS_ROOT}/{split}/labels/{lbl}", f"{dst}/{tgt}/labels/{lbl}")
    names = yaml.safe_load(open(f"{DS_ROOT}/data.yaml"))["names"]
    dyaml = f"{dst}/data.yaml"
    yaml.dump({"path": dst, "train": "train/images", "val": "val/images", "test": "test/images",
               "nc": len(names), "names": names}, open(dyaml, "w"))
    return dyaml


In [ ]:
# ---------- 3 run LOSO ----------
from ultralytics import YOLO
from collections import Counter
import pandas as pd
EPOCHS = 1 if QUICK_TEST else 15
MIN_BOX = 50   # lớp dưới ngưỡng này trong test → NaN, loại khỏi mAP trung bình của run
rows = []
for held in ["A", "B", "C"]:
    key = f"e7_loso_{held}"
    if done(key): rows.append(STATE[key]); print("skip:", key); continue
    dyaml = build_loso(f"ds_loso_{held}", held)
    # đếm box test theo lớp để lọc ngưỡng
    cnt = Counter()
    for lf in glob.glob(f"{WORK}/ds_loso_{held}/test/labels/*.txt"):
        for line in open(lf):
            if line.strip(): cnt[int(line.split()[0])] += 1
    t0 = time.time()
    m = YOLO("yolo26s.pt")
    m.train(data=dyaml, epochs=EPOCHS, imgsz=640, batch=16, seed=SEED, lr0=0.01,
            project=f"{WORK}/runs", name=key, exist_ok=True, plots=False, verbose=False)
    t = m.val(data=dyaml, split="test", batch=16)
    names = list(t.names.values())
    valid, per = [], {}
    for i, (n, a) in enumerate(zip(names, t.box.ap50)):
        if cnt.get(i, 0) >= MIN_BOX:
            per[f"AP50_{n}"] = round(float(a), 4); valid.append(float(a))
    rec = {"held_out": held, "n_test_img": len(glob.glob(f"{WORK}/ds_loso_{held}/test/images/*")),
           "mAP50_valid_classes": round(sum(valid) / max(len(valid), 1), 4),
           "n_valid_classes": len(valid), **per,
           "min": round((time.time() - t0) / 60, 1)}
    STATE[key] = rec; save_state(); rows.append(rec)
    shutil.rmtree(f"{WORK}/ds_loso_{held}", ignore_errors=True)
df = pd.DataFrame(rows); df.to_csv(f"{WORK}/results/e7_loso.csv", index=False)
print(df.to_string(index=False))


In [ ]:
# ==================== DỌN OUTPUT: KHÔNG kèm dataset ====================
KEEP = {"results", "state.json", "runs"}
for item in os.listdir(WORK):
    if item in KEEP: continue
    p = os.path.join(WORK, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
print("Output cuối:", sorted(os.listdir(WORK)))
